# Agenda 

The goal of this notebook is to ensure eveything is setup for the workshop

## Setup

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import dotenv
import os
import requests
import rich
import uuid

In [3]:
dotenv.load_dotenv("../env_workshop")

True

In [4]:
using_own_keys_flag = os.environ.get("USING_OWN_KEYS") == "true"

In [5]:
if using_own_keys_flag:
    print ("will try to use user provided keys")

    assert os.environ.get("OPENAI_API_KEY") is not None, "Please set OPENAI_API_KEY in ../env_workshop"
    assert os.environ.get("ARIZE_API_KEY") is not None, "Please set ARIZE_API_KEY in ../env_workshop"
else:
    print ("will use proxsy server, so no need for user provided keys")

will use proxsy server, so no need for user provided keys


## Openai validation

In [6]:
from langchain_openai import ChatOpenAI


In [7]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")





In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=OPENAI_BASE_URL,
    #temperature=0.2,
    #max_tokens=512,
)

llm.invoke("what is the weather in seattle")

AIMessage(content="I can't provide real-time weather updates. For the latest weather information in Seattle, I recommend checking a reliable weather website or app.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 14, 'total_tokens': 40, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CSYO06RNabrvSYgAdAkMuLmZrAnTn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--113d7ec2-9bcc-4c08-a28c-a0a9508326a5-0', usage_metadata={'input_tokens': 14, 'output_tokens': 26, 'total_tokens': 40, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

langchain has a thin wrapper around the openai client

## Tavilly

[Tavily](https://www.tavily.com/) is a service provider that enables agents to access the web

In [9]:
TAVILY_BASE_URL = os.environ.get("TAVILY_BASE_URL")
TAVILY_BASE_URL

'https://llm-proxy.np-training.dev/v1/tavily/search'

In [10]:
def tavily_search(query, **kw):
    r = requests.post(TAVILY_BASE_URL,
                      json={"query": query, **kw}, timeout=60)
    r.raise_for_status()
    return r.json()

res = tavily_search("what is the weather in seattle")
print(res)

{'query': 'what is the weather in seattle', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Seattle', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1760920743, 'localtime': '2025-10-19 17:39'}, 'current': {'last_updated_epoch': 1760920200, 'last_updated': '2025-10-19 17:30', 'temp_c': 15.6, 'temp_f': 60.1, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 5.4, 'wind_kph': 8.6, 'wind_degree': 231, 'wind_dir': 'SW', 'pressure_mb': 1014.0, 'pressure_in': 29.94, 'precip_mm': 0.02, 'precip_in': 0.0, 'humidity': 57, 'cloud': 75, 'feelslike_c': 15.6, 'feelslike_f': 60.1, 'windchill_c': 13.9, 'windchill_f': 57.0, 'heatindex_c': 14.1, 'heatindex_f': 57.4, 'dewpoint_c': 12.0, 'dewpoin

In [11]:
rich.print(res)

{
    'query': 'what is the weather in seattle',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'title': 'Weather in Seattle',
            'url': 'https://www.weatherapi.com/',
            'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of 
America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1760920743, 
'localtime': '2025-10-19 17:39'}, 'current': {'last_updated_epoch': 1760920200, 'last_updated': '2025-10-19 17:30',
'temp_c': 15.6, 'temp_f': 60.1, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': 
'//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 5.4, 'wind_kph': 8.6, 'wind_degree': 
231, 'wind_dir': 'SW', 'pressure_mb': 1014.0, 'pressure_in': 29.94, 'precip_mm': 0.02, 'precip_in': 0.0, 
'humidity': 57, 'cloud': 75, 'feelslike_c': 15.6, 'feelslike_f': 60.1, 'windchill_c': 13.9, 'windchill_f': 57.0, 
'heatindex_c': 14.1, 'heatindex_f': 57.4, 'dewpoint_c': 12.0, 'dewpoint_f': 53.5, 'vis_km': 16.0, 'vis_miles': 9.0,
'uv': 0.3, 'gust_mph': 8.3, 'gust_kph': 13.4}}",
            'score': 0.9609954,
            'raw_content': None
        },
        {
            'url': 'https://weathershogun.com/weather/usa/wa/seattle/4787/october/2025-10-20',
            'title': 'Monday, October 20, 2025. Seattle, WA - Weather Forecast',
            'content': 'Seattle, Washington Weather: Monday, October 20, 2025. Day 59°. Night 48°. Precipitation 3 
%. Wind 4 mph. UV Index (0 - 11+) 11',
            'score': 0.9477354,
            'raw_content': None
        },
        {
            'url': 'https://world-weather.info/forecast/usa/seattle/october-2025/',
            'title': 'Weather in Seattle in October 2025 (Washington)',
            'content': '*   [10 +59° +52° 7.6 mph S 29.6 inHg 61 %07:21 am 06:30 
pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-10) *   [11 +57° +52° 5.4 mph S 29.7 inHg 89 
%07:23 am 06:28 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-11) *   [12 +54° +50° 10.5 mph 
S 29.6 inHg 64 %07:24 am 06:26 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-12) *   [13 +59°
+48° 8.9 mph N 29.7 inHg 46 %07:26 am 06:24 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-13)
*   [14 +61° +48° 5.8 mph N 29.8 inHg 46 %07:27 am 06:22 
pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-14) *   [17 +59° +48° 4.7 mph N 29.5 inHg 61 
%07:31 am 06:17 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-17) *   [18 +59° +48° 7.8 mph S
29.7 inHg 60 %07:33 am 06:15 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-18) *   [20 +48° 
+48° 18.6 mph S 30.1 inHg 81 %07:36 am 06:11 
pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-20) *   [22 +45° +43° 15.2 mph S 30.3 inHg 77 
%07:39 am 06:08 pm](https://world-weather.info/forecast/usa/seattle/14days/#2025-10-22)',
            'score': 0.83093774,
            'raw_content': None
        },
        {
            'url': 
'https://en.climate-data.org/north-america/united-states-of-america/washington/seattle-593/t/october-10/',
            'title': 'Weather Seattle in October 2025: Temperature & Climate',
            'content': 'In Seattle, the temperatures in October are quite mild. The average temperature is around 
52°F (11.1°C), making it a pleasant month for outdoor activities',
            'score': 0.81630427,
            'raw_content': None
        },
        {
            'url': 
'https://www.easeweather.com/north-america/united-states/washington/king-county/seattle/october',
            'title': 'Weather in Seattle in October 2025 - Detailed Forecast - EaseWeather',
            'content': '# Weather in Seattle, Washington for October 2025 Your guide to Seattle weather in October 
- trends and predictions * Seattle experiences **heavy rainfall** in October, with over 20

In [12]:
res = tavily_search("what is the best running shoes")
rich.print(res)

{
    'query': 'what is the best running shoes',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'url': 
'https://www.fleetfeet.com/running-shoe-buyers-guide?srsltid=AfmBOoqCmJYRpFvpvZoI-2FgSp-udcZWJrK8WBaesW4ETZ_UGU8_mg
4Q',
            'title': 'The 10 Best Running Shoes of 2025 | Tested & Reviewed - Fleet Feet',
            'content': 'The HOKA Clifton 10 is one the best running shoes for anyone who demands plush cushioning, 
shock absorption and a roomy fit. In classic HOKA',
            'score': 0.8652358,
            'raw_content': None
        },
        {
            'url': 'https://www.youtube.com/watch?v=MFpzCcf6Q9U',
            'title': 'The Best Running Shoe From Every Brand (100% honest review)',
            'content': 'The Best Running Shoe From Every Brand (100% honest review)\nBen Parkes\n300000 
subscribers\n8678 likes\n630355 views\n2 Jun 2025\nIf you enjoyed the video, please like, comment and subscribe! 
Thank you for watching!\n\nSave 10% site wide on training plans, hats, technical & casual apparel - (Code - 
YOUTUBE10) \nhttps://bit.ly/benparkesyoutube10\n\nCheck out:\nTraining Plans https://bit.ly/benparkesplans\nRunning
Hats https://bit.ly/benparkeshats\nTechnical Gear https://bit.ly/benparkestechgear\nFree Beginner Plans 
https://bit.ly/benparkesfreeplans\nShop our latest arrivals https://bit.ly/47MgROg\n\nFollow me here:\nStrava: 
https://www.strava.com/athletes/2310069\nInstagram: https://www.instagram.com/benparkes\nSecond channel 
https://www.youtube.com/@benparkestheextramile\n\n📱For support enquiries or business enquiries, please go to: 
https://www.benparkes.com and use the chat function. \n\n⏰ Timecodes ⏰\n0:00 Intro\n0:53 Nike\n2:49 Asics\n4:50 
New Balance\n6:59 Hoka\n9:22 Puma\n11:32 Saucony\n13:27 Brooks\n15:33 Mizuno\n17:28 On\n19:48 Adidas\n22:16 What 
would you pick?\n\nAbout this video: In today’s video I’m choosing what I think is the best shoe from every running
shoe brand here in 2025. Not necessarily the fastest or most expensive, but the ones which standout above all the 
others in the line up that offer something really unique and special. From race-day supershoes to high-mileage 
daily trainers and ultra-cushioned recovery shoes, this is your ultimate running shoe guide so whether you’re a 
beginner or training for your next marathon, there’s something in here for you!\n\nThis video is NOT sponsored. All
products mentioned have been bought 100% by Ben and the channel. Our mission here is to help runners of all 
abilities improve and enjoy running to their full potential. Showcasing the best running tips and advice, while 
reviewing the latest gear and taking your round some of the best races in the world!\n422 comments\n',
            'score': 0.8506799,
            'raw_content': None
        },
        {
            'url': 'https://theruntesters.com/running-shoes/the-best-running-shoes-to-buy/',
            'title': 'The Best Running Shoes 2025 - The Run Testers',
            'content': 'Skip to content # The Best Running Shoes 2025 The Adidas Adizero Evo SL is one of the best 
shoes we’ve tested in several years and offers outstanding value for money as well as a high level of performance, 
to the point where it also topped the charts in our best-allrounder category. ## Puma Velocity Nitro 4 Value is 
also an important aspect of beginner shoes, so we’d recommend looking for long running and popular lines of shoes 
that are often on sale, that will suit most runners. This usually involves picking our favourite racing shoe, daily
trainer and cushioned shoe, though we often try and sneak in a fourth shoe as well. + Report this content',
            'score': 0.8409005,
            'raw_content': None
        },
        {
            'url': 'https://www.runnersworld.com/gear/a19663621/best-running-shoes/',
            'title': "The 12 Best Running Shoes of 2025 - Runner's World",
            'content': '3. The 12 Best 

## LLM Monitoring with Arize Phoenix OTEL

It can be hard to monitor LLM calls especially when they are part of a larger workflow.

We will set up Arize Phoenix OpenTelemetry to help with that.



In [16]:
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")

In [17]:
if PHOENIX_PROJECT_NAME is None or PHOENIX_PROJECT_NAME in ("npatta01","anonymous",""):
    raise ValueError("Please set PHOENIX_PROJECT_NAME in ../env_workshop")

ValueError: Please set PHOENIX_PROJECT_NAME in ../env_workshop

In [18]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata

# configure the Phoenix tracer
tracer_provider = register(
  project_name=PHOENIX_PROJECT_NAME, 
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: npatta01
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: llm-tracing.np-training.dev:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



{'data': {'name': 'npatta01', 'description': None, 'id': 'UHJvamVjdDoz'}}


View your spans at:

https://llm-tracing.np-training.dev/projects/UHJvamVjdDoz/spans


       


In [24]:
from langchain.agents import create_agent

In [23]:
def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-4o-mini",   
    tools=[get_weather],  
    system_prompt="You are a helpful assistant"  
)

res = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in seattle"}]}
)

In [32]:
arize_project_response = requests.get(
    f"{os.environ.get('PHOENIX_COLLECTOR_ENDPOINT')}/v1/projects/{PHOENIX_PROJECT_NAME}",
)

data = arize_project_response.json()

print (data)
# View your spans at:
print (f"""
       
View your spans at:

https://llm-tracing.np-training.dev/projects/{data['data']['id']}/spans
       
       
       """)

{'data': {'name': 'npatta01', 'description': None, 'id': 'UHJvamVjdDoz'}}


View your spans at:

https://llm-tracing.np-training.dev/projects/UHJvamVjdDoz/spans


       
